#### 쿼리 라우팅 
- 하나의 벡터 저장소만 사용하는 편이 유용하더라도 필요한 데이터는 RDB 를 비롯한 다양한 벡터 저장소 등에 다양한 데이터 소스에 존재할 수 있다.
- 쿼리를 라우팅해서 적절한 데이터 소스로 전달해 관련문서를 검색한다. 
- 쿼리 라우팅은 사용자의 쿼리를 적절한 데이터 소스로 전달하는 방법이다.

In [2]:
from typing import Literal 


from langchain_core.prompts import ChatPromptTemplate 
from pydantic import BaseModel, Field 
from langchain_ollama import ChatOllama 
from langchain_core.runnables import RunnableLambda 


class RouteQuery(BaseModel): 
    '''Route a user query to the most relevant datasource.'''
    datasource : Literal['python_docs', 'js_docs'] = Field(..., 
                                                           description='Given a user question, choose which datasource would be most relevant for answering their question')
    
llm = ChatOllama(model='gemma4:latest', temperature=0)

structured_llm = llm.with_structured_output(RouteQuery)

system = '''당신은 사용자 질문을 적절한 데이터 소스로 라우팅하는 전문가입니다. 질문이 지목하는 프로그래밍 언어에 따라 해당 데이터 소스로 라우팅하세요'''

prompt = ChatPromptTemplate.from_messages(
    [('system', system), ('human', '{question}')]
)

router = prompt | structured_llm 

In [ ]:
question = '''이 코드가 안 돌아가는 이유를 설명해주세요

from langchain_cor.prompts
import ChatPromptTemplate

prompt = ChatPromptTemplate.from_message(['human', 'speak in {language}'])

prompt.invoke('french')'''

result = router.invoke({'question':question})
print('\n Routing to' , result)


 Routing to datasource='python_docs'


In [4]:
def choose_route(result):
    if 'python_docs' in result.datasource.lower():
        return 'chain for python_docs'
    else: 
        return 'chain for js_docs'
    

full_chain = router| RunnableLambda(choose_route)

result = full_chain.invoke({'question' : question})

print('\nChoose route' , result)


Choose route chain for python_docs
